In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import StandardScaler

In [2]:
BASE_DIR = Path("..")

DATA_DIR = BASE_DIR / "data" / "processed" / "week-3"

NODES_FILE = DATA_DIR / "graph_node_features.csv"
EDGES_FILE = DATA_DIR / "knn_graph_edges.csv"
TARGETS_FILE = DATA_DIR / "graph_targets.csv"

EMBEDDING_FILE = DATA_DIR / "spatial_embeddings.csv"

In [3]:
nodes_df = pd.read_csv(NODES_FILE)
edges_df = pd.read_csv(EDGES_FILE)
targets_df = pd.read_csv(TARGETS_FILE)

print("Nodes:", nodes_df.shape)
print("Edges:", edges_df.shape)
print("Targets:", targets_df.shape)

print("\nNode columns:")
print(nodes_df.columns.tolist())

print("\nEdge columns:")
print(edges_df.columns.tolist())

Nodes: (21613, 18)
Edges: (108065, 3)
Targets: (21613, 2)

Node columns:
['node_id', 'id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'House Age', 'yr_built', 'yr_renovated', 'lat', 'long']

Edge columns:
['source', 'target', 'distance_km']


In [4]:
print(nodes_df[["node_id", "lat", "long"]].head())

   node_id      lat     long
0        0  47.5112 -122.257
1        1  47.7210 -122.319
2        2  47.7379 -122.233
3        3  47.5208 -122.393
4        4  47.6168 -122.045


In [5]:
spatial_features = nodes_df[["node_id", "lat", "long"]].copy()

spatial_features.head()

,node_id,lat,long
0,0,47.5112,-122.257
1,1,47.7210,-122.319
2,2,47.7379,-122.233
3,3,47.5208,-122.393
4,4,47.6168,-122.045


In [6]:
scaler = StandardScaler()

spatial_features[["lat_scaled", "long_scaled"]] = scaler.fit_transform(spatial_features[["lat", "long"]])

spatial_features.head()

,node_id,lat,long,lat_scaled,long_scaled
0,0,47.5112,-122.257,-0.352572,-0.306079
1,1,47.7210,-122.319,1.161568,-0.746341
2,2,47.7379,-122.233,1.283537,-0.135655
3,3,47.5208,-122.393,-0.283288,-1.271816
4,4,47.6168,-122.045,0.409550,1.199335


In [7]:
distance_stats = (
    edges_df
    .groupby("source")["distance_km"]
    .agg(
        mean_neighbor_distance="mean",
        median_neighbor_distance="median",
        min_neighbor_distance="min",
        max_neighbor_distance="max",
        std_neighbor_distance="std"
    )
    .reset_index()
)

distance_stats = distance_stats.rename(columns={"source": "node_id"})

distance_stats.head()

,node_id,mean_neighbor_distance,median_neighbor_distance,min_neighbor_distance,max_neighbor_distance,std_neighbor_distance
0,0,0.084509,0.075925,0.044478,0.164361,0.047154
1,1,0.115707,0.134016,0.075627,0.152972,0.034670
2,2,0.159941,0.149975,0.143364,0.193210,0.021171
3,3,0.171827,0.208451,0.055597,0.266545,0.103646
4,4,0.102912,0.088956,0.011119,0.216119,0.074250


In [8]:
neighbor_count = (edges_df.groupby("source")["target"].nunique().reset_index())

neighbor_count.columns = ["node_id","neighbor_count"]

neighbor_count.head()

,node_id,neighbor_count
0,0,5
1,1,5
2,2,5
3,3,5
4,4,5


In [9]:
embedding_df = spatial_features.merge(distance_stats,on="node_id",how="left")

embedding_df = embedding_df.merge(neighbor_count,on="node_id",how="left")

embedding_df.head()

,node_id,lat,long,lat_scaled,long_scaled,mean_neighbor_distance,median_neighbor_distance,min_neighbor_distance,max_neighbor_distance,std_neighbor_distance,neighbor_count
0,0,47.5112,-122.257,-0.352572,-0.306079,0.084509,0.075925,0.044478,0.164361,0.047154,5
1,1,47.7210,-122.319,1.161568,-0.746341,0.115707,0.134016,0.075627,0.152972,0.034670,5
2,2,47.7379,-122.233,1.283537,-0.135655,0.159941,0.149975,0.143364,0.193210,0.021171,5
3,3,47.5208,-122.393,-0.283288,-1.271816,0.171827,0.208451,0.055597,0.266545,0.103646,5
4,4,47.6168,-122.045,0.409550,1.199335,0.102912,0.088956,0.011119,0.216119,0.074250,5


In [10]:
embedding_columns = [
    "lat_scaled",
    "long_scaled",
    "mean_neighbor_distance",
    "median_neighbor_distance",
    "min_neighbor_distance",
    "max_neighbor_distance",
    "std_neighbor_distance",
    "neighbor_count"
]

embedding_df[embedding_columns] = (embedding_df[embedding_columns].replace([np.inf, -np.inf], np.nan))

embedding_df[embedding_columns] = (embedding_df[embedding_columns].fillna(0))

print("Remaining missing values:",embedding_df[embedding_columns].isnull().sum().sum())

Remaining missing values: 0


In [11]:
neighborhood_columns = [
    "mean_neighbor_distance",
    "median_neighbor_distance",
    "min_neighbor_distance",
    "max_neighbor_distance",
    "std_neighbor_distance"
]

neighborhood_scaler = StandardScaler()

embedding_df[[f"{col}_scaled" for col in neighborhood_columns]] = neighborhood_scaler.fit_transform(embedding_df[neighborhood_columns])

embedding_df.head()

,node_id,lat,long,lat_scaled,long_scaled,mean_neighbor_distance,median_neighbor_distance,min_neighbor_distance,max_neighbor_distance,std_neighbor_distance,neighbor_count,mean_neighbor_distance_scaled,median_neighbor_distance_scaled,min_neighbor_distance_scaled,max_neighbor_distance_scaled,std_neighbor_distance_scaled
0,0,47.5112,-122.257,-0.352572,-0.306079,0.084509,0.075925,0.044478,0.164361,0.047154,5,-0.393805,-0.432609,-0.235976,-0.330631,-0.310902
1,1,47.7210,-122.319,1.161568,-0.746341,0.115707,0.134016,0.075627,0.152972,0.034670,5,-0.283926,-0.235641,-0.106830,-0.364347,-0.462616
2,2,47.7379,-122.233,1.283537,-0.135655,0.159941,0.149975,0.143364,0.193210,0.021171,5,-0.128139,-0.181530,0.174005,-0.245233,-0.626666
3,3,47.5208,-122.393,-0.283288,-1.271816,0.171827,0.208451,0.055597,0.266545,0.103646,5,-0.086277,0.016743,-0.189874,-0.028148,0.375635
4,4,47.6168,-122.045,0.409550,1.199335,0.102912,0.088956,0.011119,0.216119,0.074250,5,-0.328989,-0.388425,-0.374280,-0.177417,0.018386


In [12]:
final_embedding_columns = [
    "lat_scaled",
    "long_scaled",
    "mean_neighbor_distance_scaled",
    "median_neighbor_distance_scaled",
    "min_neighbor_distance_scaled",
    "max_neighbor_distance_scaled",
    "std_neighbor_distance_scaled",
    "neighbor_count"
]

spatial_embeddings = embedding_df[["node_id"] + final_embedding_columns].copy()

spatial_embeddings.head()

,node_id,lat_scaled,long_scaled,mean_neighbor_distance_scaled,median_neighbor_distance_scaled,min_neighbor_distance_scaled,max_neighbor_distance_scaled,std_neighbor_distance_scaled,neighbor_count
0,0,-0.352572,-0.306079,-0.393805,-0.432609,-0.235976,-0.330631,-0.310902,5
1,1,1.161568,-0.746341,-0.283926,-0.235641,-0.106830,-0.364347,-0.462616,5
2,2,1.283537,-0.135655,-0.128139,-0.181530,0.174005,-0.245233,-0.626666,5
3,3,-0.283288,-1.271816,-0.086277,0.016743,-0.189874,-0.028148,0.375635,5
4,4,0.409550,1.199335,-0.328989,-0.388425,-0.374280,-0.177417,0.018386,5


In [13]:
print("Embedding shape:", spatial_embeddings.shape)

print("\nMissing values:")
print(spatial_embeddings.isnull().sum())

print("\nDuplicate node IDs:")
print(spatial_embeddings["node_id"].duplicated().sum())

Embedding shape: (21613, 9)

Missing values:
node_id                            0
lat_scaled                         0
long_scaled                        0
mean_neighbor_distance_scaled      0
median_neighbor_distance_scaled    0
min_neighbor_distance_scaled       0
max_neighbor_distance_scaled       0
std_neighbor_distance_scaled       0
neighbor_count                     0
dtype: int64

Duplicate node IDs:
0


In [14]:
spatial_embeddings[final_embedding_columns].describe()

,lat_scaled,long_scaled,mean_neighbor_distance_scaled,median_neighbor_distance_scaled,min_neighbor_distance_scaled,max_neighbor_distance_scaled,std_neighbor_distance_scaled,neighbor_count
count,2.161300e+04,2.161300e+04,2.161300e+04,2.161300e+04,2.161300e+04,2.161300e+04,2.161300e+04,21613.0
mean,1.695943e-14,-3.667318e-14,9.205199e-18,-1.144075e-16,1.578034e-17,-3.024565e-17,4.799854e-17,5.0
std,1.000023e+00,1.000023e+00,1.000023e+00,1.000023e+00,1.000023e+00,1.000023e+00,1.000023e+00,0.0
min,-2.916795e+00,-2.166543e+00,-6.679391e-01,-6.900464e-01,-4.203819e-01,-7.842569e-01,-8.794807e-01,5.0
25%,-6.426977e-01,-8.102505e-01,-2.934444e-01,-3.226120e-01,-2.359759e-01,-3.433826e-01,-4.330407e-01,5.0
50%,8.478232e-02,-1.143518e-01,-1.467521e-01,-1.588848e-01,-9.685168e-02,-1.531401e-01,-2.004041e-01,5.0
75%,8.512345e-01,6.312541e-01,7.099637e-02,8.059180e-02,8.673463e-02,7.868167e-02,1.226692e-01,5.0
max,1.570054e+00,6.383070e+00,8.099575e+01,7.851232e+01,9.351559e+01,6.880058e+01,4.340282e+01,5.0


In [15]:
spatial_embeddings.to_csv(EMBEDDING_FILE,index=False)

print("✅ Spatial embeddings saved successfully.")
print(f"📁 {EMBEDDING_FILE}")

✅ Spatial embeddings saved successfully.
📁 ..\data\processed\week-3\spatial_embeddings.csv


In [16]:
saved_embeddings = pd.read_csv(EMBEDDING_FILE)

print("Saved embedding shape:")
print(saved_embeddings.shape)

print("\nColumns:")
print(saved_embeddings.columns.tolist())

Saved embedding shape:
(21613, 9)

Columns:
['node_id', 'lat_scaled', 'long_scaled', 'mean_neighbor_distance_scaled', 'median_neighbor_distance_scaled', 'min_neighbor_distance_scaled', 'max_neighbor_distance_scaled', 'std_neighbor_distance_scaled', 'neighbor_count']


In [17]:
assert len(saved_embeddings) == len(nodes_df)

assert saved_embeddings["node_id"].nunique() == len(nodes_df)

assert saved_embeddings.isnull().sum().sum() == 0

assert saved_embeddings["node_id"].duplicated().sum() == 0

print("✅ Spatial embedding validation passed.")

✅ Spatial embedding validation passed.


In [18]:
print("=" * 55)
print("WEEK 3 DAY 3 – SPATIAL EMBEDDING SUMMARY")
print("=" * 55)

print(f"Number of nodes: {len(spatial_embeddings):,}")
print(f"Embedding dimensions: {len(final_embedding_columns)}")
print(f"Missing values: {spatial_embeddings.isnull().sum().sum()}")
print(
    f"Duplicate node IDs: "
    f"{spatial_embeddings['node_id'].duplicated().sum()}"
)

print("\nEmbedding features:")
for feature in final_embedding_columns:
    print(f"- {feature}")

print("\n✅ Spatial embedding generation completed successfully.")

WEEK 3 DAY 3 – SPATIAL EMBEDDING SUMMARY
Number of nodes: 21,613
Embedding dimensions: 8
Missing values: 0
Duplicate node IDs: 0

Embedding features:
- lat_scaled
- long_scaled
- mean_neighbor_distance_scaled
- median_neighbor_distance_scaled
- min_neighbor_distance_scaled
- max_neighbor_distance_scaled
- std_neighbor_distance_scaled
- neighbor_count

✅ Spatial embedding generation completed successfully.
